# Experiment: template-switch candidate analysis

**Objective:** identify UMI/sample combinations linked to multiple corrected barcode representatives and test how a minimum per-barcode read threshold changes candidate calls.

**Success criteria:** recover the deliberately shared synthetic UMI, retain per-barcode support, and avoid interpreting association as proof of mechanism.

## Plan and hypotheses

1. Query barcode-corrected but pre-UMI-collapse observations.
2. Count distinct barcode representatives per `(sample_index, UMI)`.
3. Inspect support for each barcode in multi-barcode groups.
4. Apply a lenient minimum support threshold.

**Hypothesis:** the synthetic shared UMI remains a candidate because both distant barcodes have multiple supporting reads. A one-read association would be more consistent with noise.

In [ ]:
from __future__ import annotations

import json
import sqlite3
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
database = ROOT / 'examples' / 'output' / 'example.sqlite'
if not database.exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic_mapseq.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_example_pipeline.py')], check=True)
truth = json.loads((ROOT / 'examples' / 'synthetic' / 'truth.json').read_text())
con = sqlite3.connect(database)

## Minimal baseline: count barcode representatives per UMI

Use `collapsed_counts`, not `umi_collapsed_counts`: the question is which raw observed UMIs connect multiple corrected barcodes before UMI neighbors are merged.

In [ ]:
groups = con.execute('''
SELECT sample_index, umi, count(DISTINCT barcode) AS n_barcodes, sum(read_count) AS reads
FROM collapsed_counts
GROUP BY sample_index, umi
ORDER BY n_barcodes DESC, reads DESC
''').fetchall()
distribution = Counter(row[2] for row in groups)
print('Number of barcode representatives per UMI:', dict(sorted(distribution.items())))
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(list(sorted(distribution)), [distribution[key] for key in sorted(distribution)], color='#6DCBF4', edgecolor='black')
ax.set(xlabel='Distinct barcode representatives per UMI', ylabel='UMI/sample groups', title='Multiplicity of barcode associations')
plt.tight_layout()
plt.show()

## Inspect candidate support

A candidate is more credible when each associated barcode has more than one supporting read. The query below preserves barcode-level support instead of reducing immediately to one percentage.

In [ ]:
candidates = con.execute('''
WITH multi AS (
  SELECT sample_index, umi
  FROM collapsed_counts
  GROUP BY sample_index, umi
  HAVING count(DISTINCT barcode) > 1
)
SELECT c.sample_index, c.umi, c.barcode, c.read_count
FROM collapsed_counts c
JOIN multi m USING(sample_index, umi)
ORDER BY c.sample_index, c.umi, c.read_count DESC
''').fetchall()
print('All multi-barcode UMI rows:')
for row in candidates:
    print(row)

MIN_READS_PER_BARCODE_UMI = 2
supported = [row for row in candidates if row[3] >= MIN_READS_PER_BARCODE_UMI]
supported_groups = Counter((row[0], row[1]) for row in supported)
supported_candidates = {key: count for key, count in supported_groups.items() if count > 1}
print('Supported candidate groups:', supported_candidates)

In [ ]:
expected = truth['template_switch_candidate']
expected_key = (expected['sample_index'], expected['umi'])
assert expected_key in supported_candidates
print('Recovered synthetic positive control:', expected_key)
con.close()

## Result, exercise, and next steps

The deliberately shared UMI is recovered with multi-read support for both barcodes. This demonstrates detection, not biological causality: UMI collision, index error, barcode over-collapsing, and true template switching remain alternative explanations.

**Exercise:** sweep `MIN_READS_PER_BARCODE_UMI` from 1 to 5 and record the candidate count.

**Next steps for real data:** stratify by sample, compare injection-site versus distal regions, inspect barcode abundance imbalance, and report both the fraction of UMIs and fraction of reads involved.